# Examen del segundo corte
### Matemáticas para Inteligencia Artificial

**Desarrollo de los reactivos IV, V y VI**

Cada ejercicio contiene su planteamiento, el razonamiento matemático, la comprobación con Python y la interpretación de los resultados. Las ecuaciones se presentan con notación matemática convencional; el código se utiliza como herramienta de verificación y graficación.


In [ ]:
import numpy as np
import sympy as sp
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Math

np.set_printoptions(precision=10, suppress=True)

def notacion_cientifica(valor, decimales=4):
    """Da formato matemático a un resultado en notación científica."""
    if valor == 0:
        return "0"
    mantisa, exponente = f"{valor:.{decimales}e}".split("e")
    return rf"{mantisa}\times 10^{{{int(exponente)}}}"

# IV. Optimización con restricciones y programación lineal

## IV.A. Multiplicadores de Lagrange

Se busca minimizar la función

$$
C(x,y)=x^{2}+2y^{2},
\qquad \text{sujeta a}\qquad x+y=6.
$$

### IV.A.1. Planteamiento y solución analítica

Como existe una restricción de igualdad, introducimos un multiplicador $\lambda$ y escribimos la función lagrangiana siguiendo la convención del enunciado:

$$
\mathcal{L}(x,y,\lambda)=x^{2}+2y^{2}-\lambda(x+y-6).
$$

Derivamos con respecto a las tres variables e igualamos a cero:

$$
\begin{aligned}
\frac{\partial \mathcal{L}}{\partial x}&=2x-\lambda=0,\\[4pt]
\frac{\partial \mathcal{L}}{\partial y}&=4y-\lambda=0,\\[4pt]
\frac{\partial \mathcal{L}}{\partial \lambda}&=-(x+y-6)=0.
\end{aligned}
$$

Las dos primeras ecuaciones indican que $2x=\lambda=4y$. Así, $x=2y$. Sustituyendo en la restricción:

$$
\begin{aligned}
x+y&=6,\\
2y+y&=6,\\
3y&=6
\quad\Longrightarrow\quad y^*=2,\\[3pt]
x^*&=2y^*=4.
\end{aligned}
$$

El multiplicador y el valor óptimo se calculan directamente:

$$
\begin{aligned}
\lambda^*&=2x^*=2(4)=8,\\
C^*&=C(4,2)=4^{2}+2(2^{2})=16+8=24.
\end{aligned}
$$

Por tanto,

$$
\boxed{(x^*,y^*)=(4,2),\qquad \lambda^*=8,\qquad C^*=24.}
$$

**Comprobación en Python.** Resolvemos el sistema de estacionariedad mediante SymPy.

In [ ]:
x, y, lam = sp.symbols("x y lambda", real=True)

C = x**2 + 2*y**2
L = C - lam*(x + y - 6)

ecuaciones = [
    sp.diff(L, x),
    sp.diff(L, y),
    sp.diff(L, lam)
]

solucion = sp.solve(ecuaciones, (x, y, lam), dict=True)[0]
solucion

In [ ]:
x_star = sp.simplify(solucion[x])
y_star = sp.simplify(solucion[y])
lambda_star = sp.simplify(solucion[lam])
C_star = sp.simplify(C.subs(solucion))

display(Math(r'\boxed{x^*=%s,\quad y^*=%s,\quad \lambda^*=%s,\quad C^*=%s}' % (
    sp.latex(x_star), sp.latex(y_star), sp.latex(lambda_star), sp.latex(C_star)
)))

### IV.A.2. Globalidad del mínimo y sensibilidad

Para comprobar la naturaleza del óptimo, calculamos la Hessiana de la función objetivo:

$$
\nabla^{2}C(x,y)=
\begin{pmatrix}
2 & 0\\
0 & 4
\end{pmatrix}.
$$

Sus autovalores son $2$ y $4$, ambos estrictamente positivos. Por ello, $C$ es **estrictamente convexa**. Dado que la restricción es afín, el punto obtenido es el **mínimo global restringido y único**.

Ahora sustituimos el valor fijo de la restricción por un parámetro $b$:

$$
x+y=b.
$$

Las condiciones de Lagrange mantienen la relación $x=2y$, y se obtiene

$$
\begin{aligned}
2y+y&=b &&\Longrightarrow& y^*(b)&=\frac{b}{3},\\[3pt]
x^*(b)&=2y^*(b) &&\Longrightarrow& x^*(b)&=\frac{2b}{3}.
\end{aligned}
$$

Sustituyendo estas expresiones en el costo:

$$
\begin{aligned}
V(b)&=C\bigl(x^*(b),y^*(b)\bigr)\\
&=\left(\frac{2b}{3}\right)^{\!2}
+2\left(\frac{b}{3}\right)^{\!2}\\[3pt]
&=\frac{4b^2}{9}+\frac{2b^2}{9}
=\boxed{\frac{2}{3}b^2}.
\end{aligned}
$$

La derivada de la función de valor óptimo es

$$
V'(b)=\frac{4b}{3},\qquad V'(6)=8=\lambda^*.
$$

Esto muestra que el multiplicador representa la variación marginal del costo óptimo respecto al término derecho de la restricción. Para un incremento pequeño $\Delta b=0.1$:

$$
\begin{aligned}
V(6.1)&\approx V(6)+V'(6)\,\Delta b\\
&=24+8(0.1)=\boxed{24.8}.
\end{aligned}
$$

In [ ]:
b = sp.symbols("b", real=True)
x_b, y_b = 2*b/3, b/3
V_b = sp.simplify(x_b**2 + 2*y_b**2)
dV_b = sp.diff(V_b, b)

estimado = sp.N(C_star + lambda_star*sp.Rational(1, 10))
exacto = sp.N(V_b.subs(b, sp.Rational(61, 10)))

display(Math(r'V(b)=%s,\qquad V\prime(b)=%s' % (
    sp.latex(V_b), sp.latex(dV_b)
)))
display(Math(r'\boxed{V(6.1)\approx %.4f,\qquad V(6.1)_{\mathrm{exacto}}=%.6f}' % (
    float(estimado), float(exacto)
)))

La estimación lineal es $24.8$, mientras que el valor exacto de la función cuadrática es $24.806667$. La diferencia es pequeña porque el cambio de la restricción también es pequeño.

### IV.A.3. Comprobación por sustitución y gráfica

Podemos verificar el resultado sin introducir multiplicadores. De $x+y=6$ despejamos $x=6-y$ y sustituimos en $C$:

$$
\begin{aligned}
C(6-y,y)&=(6-y)^2+2y^2\\
&=36-12y+3y^2,\\[3pt]
\frac{dC}{dy}&=6y-12.
\end{aligned}
$$

Al imponer $\tfrac{dC}{dy}=0$ obtenemos $y=2$ y, en consecuencia, $x=4$. Como la segunda derivada es $6>0$, el resultado confirma el mínimo sobre la recta factible.

In [ ]:
C_sust = sp.expand(C.subs(x, 6-y))
dC_sust = sp.diff(C_sust, y)
y_sust = sp.solve(dC_sust, y)[0]
x_sust = sp.simplify(6-y_sust)

display(Math(r'C(6-y,y)=%s,\qquad \frac{dC}{dy}=%s' % (
    sp.latex(C_sust), sp.latex(dC_sust)
)))
display(Math(r'\boxed{(x^*,y^*)=(%s,%s)}' % (
    sp.latex(x_sust), sp.latex(y_sust)
)))

La siguiente figura muestra las curvas de nivel de $C(x,y)$, la recta de restricción y el punto óptimo. Geométricamente, el mínimo restringido ocurre donde una curva de nivel de menor valor toca a la recta factible.

In [ ]:
x_vals = np.linspace(-1, 7, 400)
y_vals = np.linspace(-1, 7, 400)
Xg, Yg = np.meshgrid(x_vals, y_vals)
Cg = Xg**2 + 2*Yg**2

plt.figure(figsize=(8, 6))
plt.contour(Xg, Yg, Cg, levels=[8, 16, 24, 32, 48, 72])
plt.plot(x_vals, 6-x_vals, label=r"$x+y=6$")
plt.scatter([4], [2], s=70, label=r"Óptimo $(4,2)$")
plt.xlabel("x")
plt.ylabel("y")
plt.title("IV.A — Curvas de nivel y restricción")
plt.xlim(-1, 7)
plt.ylim(-1, 7)
plt.grid(True)
plt.legend()
plt.show()

## IV.B. Programación lineal

Se desea maximizar una función lineal:

$$
\max_{x,y}\quad z=3x+2y,
$$

sujeta a

$$
\left\{
\begin{aligned}
x+y&\leq 4,\\
x&\leq 2,\\
y&\leq 3,\\
x,y&\geq 0.
\end{aligned}
\right.
$$

### IV.B.1. Vértices de la región factible

Las fronteras de las restricciones se intersectan en los siguientes **vértices factibles**:

$$
\boxed{(0,0),\ (2,0),\ (2,2),\ (1,3),\ (0,3).}
$$

No todas las intersecciones de rectas son válidas; cada candidata debe satisfacer **todas** las desigualdades. El código siguiente enumera las intersecciones por pares, elimina duplicados y descarta las que quedan fuera de la región.

In [ ]:
from itertools import combinations

fronteras = [
    np.array([1.0, 1.0, 4.0]),  # x+y=4
    np.array([1.0, 0.0, 2.0]),  # x=2
    np.array([0.0, 1.0, 3.0]),  # y=3
    np.array([1.0, 0.0, 0.0]),  # x=0
    np.array([0.0, 1.0, 0.0]),  # y=0
]

def es_factible(p, tol=1e-9):
    x_, y_ = p
    return (
        x_ + y_ <= 4 + tol
        and x_ <= 2 + tol
        and y_ <= 3 + tol
        and x_ >= -tol
        and y_ >= -tol
    )

vertices = []

for f1, f2 in combinations(fronteras, 2):
    A = np.array([[f1[0], f1[1]], [f2[0], f2[1]]])
    rhs = np.array([f1[2], f2[2]])

    if abs(np.linalg.det(A)) < 1e-12:
        continue

    p = np.linalg.solve(A, rhs)

    if es_factible(p) and not any(np.allclose(p, q) for q in vertices):
        vertices.append(p)

centro = np.mean(vertices, axis=0)
vertices = sorted(
    vertices,
    key=lambda p: np.arctan2(p[1]-centro[1], p[0]-centro[0])
)

vertices

### IV.B.2. Evaluación de la función objetivo

Evaluamos $z=3x+2y$ en los cinco vértices:

$$
\begin{array}{c|c}
\text{Vértice }(x,y)&z=3x+2y\\\hline
(0,0)&0\\
(2,0)&6\\
(2,2)&10\\
(1,3)&9\\
(0,3)&6
\end{array}
$$

El valor mayor se obtiene en $(2,2)$:

$$
\boxed{(x^*,y^*)=(2,2),\qquad z_{\max}=10.}
$$

La enumeración computacional permite comprobar tanto los vértices como el valor del máximo.

In [ ]:
resultados_lp = []
for p in vertices:
    x_, y_ = p
    resultados_lp.append((float(x_), float(y_), float(3*x_ + 2*y_)))

optimo_lp = max(resultados_lp, key=lambda t: t[2])
tabla_vertices = pd.DataFrame(resultados_lp, columns=["x", "y", "z = 3x + 2y"])
display(tabla_vertices.style.format(precision=1))
display(Math(r'\boxed{(x^*,y^*)=(%.0f,%.0f),\qquad z_{\max}=%.0f}' % optimo_lp))

In [ ]:
verts = np.array(vertices)

plt.figure(figsize=(8, 6))
plt.fill(verts[:, 0], verts[:, 1], alpha=0.25, label="Región factible")
plt.plot(
    np.r_[verts[:, 0], verts[0, 0]],
    np.r_[verts[:, 1], verts[0, 1]]
)
plt.scatter(verts[:, 0], verts[:, 1], label="Vértices")

for x_, y_ in verts:
    plt.annotate(f"({x_:g},{y_:g})", (x_, y_), xytext=(5,5), textcoords="offset points")

xx = np.linspace(0, 4, 200)
yy = (10 - 3*xx)/2
plt.plot(xx, yy, label=r"$3x+2y=10$")

plt.scatter([2], [2], s=70, label=r"Óptimo $(2,2)$")
plt.xlabel("x")
plt.ylabel("y")
plt.title("IV.B — Región factible")
plt.xlim(-0.25, 4.25)
plt.ylim(-0.25, 4.25)
plt.grid(True)
plt.legend()
plt.show()

### IV.B.3. Justificación y clasificación

Se trata de un problema de **programación lineal (LP)**: la función objetivo y las restricciones son lineales.

Como la región factible es un polígono no vacío y acotado, existe un vértice donde se alcanza el valor máximo de una función lineal. La comparación anterior muestra que $(2,2)$ es el **único vértice** con valor $10$; todos los demás valores son menores. Por tanto, el máximo es **único**.

# V. Análisis numérico y regularización

## V.A. Truncamiento, redondeo y orden de convergencia

Sea $g(x)=\ln(1+x)$. Su derivada es

$$
g'(x)=\frac{1}{1+x},\qquad\boxed{g'(1)=\frac12}.
$$

Para aproximarla se utilizan las diferencias **hacia adelante** y **central**:

$$
\begin{aligned}
D_h^+&=\frac{g(1+h)-g(1)}{h},\\[6pt]
D_h^0&=\frac{g(1+h)-g(1-h)}{2h}.
\end{aligned}
$$

### V.A.1. Término principal del error por Taylor

Para la fórmula hacia adelante, Taylor alrededor de $x=1$ da

$$
g(1+h)=g(1)+hg'(1)+\frac{h^2}{2}g''(1)+O(h^3).
$$

Restamos $g(1)$ y dividimos entre $h$:

$$
\begin{aligned}
D_h^+&=g'(1)+\frac{h}{2}g''(1)+O(h^2),\\[3pt]
D_h^+-g'(1)&=-\frac{h}{8}+O(h^2),
\qquad g''(1)=-\frac14.
\end{aligned}
$$

El término dominante es proporcional a $h$, así que el error de truncamiento es de **primer orden**:

$$
\boxed{|D_h^+-g'(1)|=O(h)}.
$$

Para la diferencia central escribimos Taylor con $+h$ y $-h$. Al restar, desaparecen los términos de potencias pares:

$$
\begin{aligned}
g(1+h)-g(1-h)
&=2hg'(1)+\frac{h^3}{3}g'''(1)+O(h^5).
\end{aligned}
$$

Dividiendo entre $2h$:

$$
\begin{aligned}
D_h^0&=g'(1)+\frac{h^2}{6}g'''(1)+O(h^4),\\[3pt]
D_h^0-g'(1)&=\frac{h^2}{24}+O(h^4),
\qquad g'''(1)=\frac14.
\end{aligned}
$$

El término dominante es proporcional a $h^2$, por lo que esta fórmula es de **segundo orden**:

$$
\boxed{|D_h^0-g'(1)|=O(h^2)}.
$$

In [ ]:
def g(x):
    return np.log1p(x)

def D_mas(h):
    return (g(1.0 + h) - g(1.0)) / h

def D_central(h):
    return (g(1.0 + h) - g(1.0 - h)) / (2*h)

exacto = 0.5
tabla_va = []
error_mas_anterior = None
error_central_anterior = None

for k in range(1, 9):
    h = 10.0**(-k)
    dm, dc = D_mas(h), D_central(h)
    em, ec = abs(dm-exacto), abs(dc-exacto)
    pm = np.nan if error_mas_anterior is None else np.log(error_mas_anterior/em)/np.log(10)
    pc = np.nan if error_central_anterior is None else np.log(error_central_anterior/ec)/np.log(10)
    tabla_va.append((k,h,dm,dc,em,ec,pm,pc))
    error_mas_anterior, error_central_anterior = em, ec

columnas = ["k", "h", "D⁺ₕ", "D⁰ₕ", "Error D⁺ₕ", "Error D⁰ₕ", "p⁺", "p⁰"]
df_va = pd.DataFrame(tabla_va, columns=columnas)
display(df_va.style.format({
    "h": "{:.0e}", "D⁺ₕ": "{:.12f}", "D⁰ₕ": "{:.12f}",
    "Error D⁺ₕ": "{:.3e}", "Error D⁰ₕ": "{:.3e}",
    "p⁺": "{:.3f}", "p⁰": "{:.3f}"
}, na_rep="—"))

### V.A.2. Orden empírico

El orden empírico se obtiene comparando errores al dividir el paso entre $10$:

$$
p=\frac{\ln\!\left(E(10h)/E(h)\right)}{\ln(10)}.
$$

La tabla confirma el comportamiento previsto por Taylor: para la fórmula hacia adelante, $p$ se mantiene cercano a **1** en el rango donde domina el truncamiento. En la diferencia central, el orden es aproximadamente **2** entre $h=10^{-1}$ y $h=10^{-4}$.

Con pasos más pequeños, los errores de precisión finita comienzan a afectar la estimación del orden, especialmente en la fórmula central.

In [ ]:
h_vals = np.array([fila[1] for fila in tabla_va])
e_mas = np.array([fila[4] for fila in tabla_va])
e_central = np.array([fila[5] for fila in tabla_va])

plt.figure(figsize=(8, 6))
plt.loglog(h_vals, e_mas, marker="o", label=r"Error de $D_h^+$")
plt.loglog(h_vals, e_central, marker="o", label=r"Error de $D_h^0$")
plt.loglog(h_vals, h_vals/8, label=r"Referencia $\propto h$")
plt.loglog(h_vals, h_vals**2/24, label=r"Referencia $\propto h^2$")
plt.xlabel("h")
plt.ylabel("Error absoluto")
plt.title("V.A — Error de las aproximaciones")
plt.grid(True)
plt.legend()
plt.show()

### V.A.3. Interpretación de la gráfica

Disminuir $h$ reduce el error de truncamiento, pero obliga a restar dos valores de $g$ cada vez más próximos. Por ejemplo,

$$
g(1+h)-g(1).
$$

En `float64` esta resta puede perder cifras significativas por **cancelación**. Además, cualquier error de redondeo del numerador se divide entre $h$, de modo que su efecto puede crecer si el paso es muy pequeño.

En la gráfica log–log se distinguen dos comportamientos: primero, el error desciende aproximadamente como $h$ o $h^2$; después, deja de mejorar e incluso aumenta. El mejor paso es resultado del compromiso entre **truncamiento y redondeo**, no simplemente el más pequeño disponible.

## V.B. Condicionamiento y efecto de Ridge

Consideramos el sistema $A_\varepsilon\mathbf{x}=\mathbf{b}$ y una perturbación $\delta\mathbf{b}$:

$$
A_\varepsilon=
\begin{pmatrix}
1&1\\
1&1+\varepsilon
\end{pmatrix},\quad
\varepsilon=10^{-8},\quad
\mathbf b=\begin{pmatrix}2\\2+\varepsilon\end{pmatrix},\quad
\delta\mathbf b=\begin{pmatrix}0\\10^{-6}\end{pmatrix}.
$$

### V.B.1. Solución simbólica y número de condición

Al desarrollar el sistema original se obtiene

$$
\left\{
\begin{aligned}
x_1+x_2&=2,\\
x_1+(1+\varepsilon)x_2&=2+\varepsilon.
\end{aligned}
\right.
$$

Restamos la primera ecuación de la segunda. Esto produce $\varepsilon x_2=\varepsilon$, luego $x_2=1$ y $x_1=1$:

$$
\boxed{\mathbf{x}=\begin{pmatrix}1\\1\end{pmatrix}}.
$$

Para estudiar la sensibilidad perturbamos únicamente la segunda ecuación en una cantidad $\delta$:

$$
\begin{aligned}
\varepsilon x_2&=\varepsilon+\delta
&&\Longrightarrow&x_2&=1+\frac{\delta}{\varepsilon},\\
x_1+x_2&=2
&&\Longrightarrow&x_1&=1-\frac{\delta}{\varepsilon}.
\end{aligned}
$$

Así, la solución perturbada es

$$
\boxed{\mathbf{x}_\delta=
\begin{pmatrix}
1-\delta/\varepsilon\\
1+\delta/\varepsilon
\end{pmatrix}}.
$$

Al sustituir $\delta=10^{-6}$ y $\varepsilon=10^{-8}$ resulta $\delta/\varepsilon=100$. Por tanto,

$$
\boxed{\mathbf{x}_\delta=
\begin{pmatrix}-99\\101\end{pmatrix}}.
$$

Un cambio pequeño en los datos provoca una diferencia muy grande en la solución.

Para cuantificar esa sensibilidad, calculamos los autovalores de $A_\varepsilon$. El polinomio característico conduce a

$$
\lambda_{\pm}=
\frac{2+\varepsilon\pm\sqrt{4+\varepsilon^{2}}}{2}.
$$

Cuando $\varepsilon$ es pequeño, el mayor autovalor es aproximadamente $2$ y el menor aproximadamente $\varepsilon/2$. Puesto que la matriz es simétrica y definida positiva, su número de condición en norma $2$ es

$$
\begin{aligned}
\kappa_2(A_\varepsilon)
&=\frac{\lambda_{\max}}{\lambda_{\min}}\\[4pt]
&\approx\frac{2}{\varepsilon/2}
=\frac{4}{\varepsilon}
=\boxed{4\times10^{8}}.
\end{aligned}
$$

In [ ]:
eps, delta = sp.symbols("epsilon delta", positive=True)

A_sym = sp.Matrix([[1, 1], [1, 1+eps]])
b_sym = sp.Matrix([2, 2+eps])
b_delta_sym = sp.Matrix([2, 2+eps+delta])

x_sym = sp.simplify(A_sym.inv()*b_sym)
x_delta_sym = sp.simplify(A_sym.inv()*b_delta_sym)

print("Solución original:")
display(x_sym)

print("Solución perturbada:")
display(x_delta_sym)

print("Autovalores:")
display(A_sym.eigenvals())

### V.B.2 Comprobación numérica

Ahora se calcula el número de condición con NumPy y se comparan las soluciones original y perturbada.

In [ ]:
eps_num, delta_num = 1e-8, 1e-6
A = np.array([[1.0, 1.0], [1.0, 1.0+eps_num]], dtype=np.float64)
b = np.array([2.0, 2.0+eps_num], dtype=np.float64)
db = np.array([0.0, delta_num], dtype=np.float64)

x0 = np.linalg.solve(A, b)
xd = np.linalg.solve(A, b+db)
cond_A = np.linalg.cond(A, 2)
cambio_rel_b = np.linalg.norm(db)/np.linalg.norm(b)
cambio_rel_x = np.linalg.norm(xd-x0)/np.linalg.norm(x0)

display(Math(r'\kappa_2(A_\varepsilon)\approx %s' % notacion_cientifica(cond_A)))
display(Math(r'\mathbf{x}=\begin{pmatrix}%.8f\\%.8f\end{pmatrix},\qquad'
             r'\mathbf{x}_\delta=\begin{pmatrix}%.8f\\%.8f\end{pmatrix}' %
             (x0[0], x0[1], xd[0], xd[1])))
display(Math(r'\begin{aligned}'
             r'\|\mathbf b\|_2 &= %.9f\\'
             r'\frac{\|\delta\mathbf b\|_2}{\|\mathbf b\|_2}&=%s\\'
             r'\frac{\|\mathbf{x}_\delta-\mathbf{x}\|_2}{\|\mathbf{x}\|_2}&=%s\\'
             r'\text{Amplificación observada}&=%s'
             r'\end{aligned}' % (
                 np.linalg.norm(b), notacion_cientifica(cambio_rel_b),
                 notacion_cientifica(cambio_rel_x),
                 notacion_cientifica(cambio_rel_x/cambio_rel_b))))

El cambio relativo de los datos es del orden de $10^{-7}$, mientras que el de la solución es del orden de $10^2$. Esto demuestra la **amplificación de perturbaciones** propia de un problema mal condicionado.

No se trata de un fallo del algoritmo de `numpy.linalg.solve`: la sensibilidad está presente en el sistema matemático. La precisión finita explica las ligeras diferencias entre los valores numéricos y la solución simbólica $(-99,101)^T$.

### V.B.3. Regularización con $\lambda=0.01$

Repetimos el cálculo con una matriz regularizada:

$$
(A_\varepsilon+\lambda I)\mathbf{x}_R=\mathbf b,
\qquad \lambda=0.01.
$$

Su forma explícita es

$$
A_\varepsilon+0.01I=
\begin{pmatrix}
1.01&1\\
1&1.01+\varepsilon
\end{pmatrix}.
$$

Sumar $\lambda I$ aumenta cada autovalor en $\lambda$. En particular, el menor deja de estar próximo a cero, por lo que la sensibilidad de la solución disminuye.

In [ ]:
lambda_ridge = 0.01
A_ridge = A + lambda_ridge*np.eye(2)
x_ridge = np.linalg.solve(A_ridge, b)
xd_ridge = np.linalg.solve(A_ridge, b+db)
cond_ridge = np.linalg.cond(A_ridge, 2)
cambio_rel_x_ridge = np.linalg.norm(xd_ridge-x_ridge)/np.linalg.norm(x_ridge)

display(Math(r'\kappa_2(A_\varepsilon+\lambda I)\approx %.4f' % cond_ridge))
display(Math(r'\mathbf{x}_R=\begin{pmatrix}%.8f\\%.8f\end{pmatrix},\qquad'
             r'\mathbf{x}_{R,\delta}=\begin{pmatrix}%.8f\\%.8f\end{pmatrix}' % (
                 x_ridge[0], x_ridge[1], xd_ridge[0], xd_ridge[1])))
display(Math(r'\begin{aligned}'
             r'\text{Cambio relativo regularizado}&=%s\\'
             r'\text{Amplificación regularizada}&=%s\\'
             r'\|\mathbf{x}_R-\mathbf{x}\|_2&=%s'
             r'\end{aligned}' % (
                 notacion_cientifica(cambio_rel_x_ridge),
                 notacion_cientifica(cambio_rel_x_ridge/cambio_rel_b),
                 notacion_cientifica(np.linalg.norm(x_ridge-x0)))))

El número de condición disminuye de aproximadamente $4\times10^8$ a $201$. La solución regularizada se modifica mucho menos cuando se perturba el vector $\mathbf b$. No obstante, ya no reproduce exactamente la solución sin regularización: esa diferencia es el **sesgo introducido**.

### V.B.4. Relación con regresión Ridge

La estimación Ridge puede escribirse como

$$
\boxed{\mathbf{w}_R=
\bigl(X^{T}X+\lambda I\bigr)^{-1}X^{T}\mathbf y.}
$$

Si las columnas de $X$ son colineales o casi colineales, $X^TX$ posee autovalores nulos o muy pequeños. La regularización transforma cada autovalor $\mu_i$ en $\mu_i+\lambda$, evitando la singularidad para $\lambda>0$.

En términos prácticos, **ganamos estabilidad a cambio de introducir sesgo**: los coeficientes se contraen y la respuesta es menos sensible a pequeñas perturbaciones de los datos.

# VI. Descenso batch y minibatch SGD

Se considera una regresión lineal sin intercepto:

$$
\widehat y_i=wx_i,\qquad
L(w)=\frac14\sum_{i=1}^4(wx_i-y_i)^2.
$$

Los cuatro datos del examen son

$$
\begin{array}{c|rrrr}
i&1&2&3&4\\\hline
x_i&-1&0&1&2\\
y_i&-2&0&2&4
\end{array}
$$

## VI.1. Función cuadrática y solución analítica

Observamos que $y_i=2x_i$ para cada observación. Esto permite simplificar cada residuo antes de desarrollar la suma:

$$
wx_i-y_i=wx_i-2x_i=(w-2)x_i.
$$

Sustituyendo en la pérdida,

$$
\begin{aligned}
L(w)&=\frac14\sum_{i=1}^4(w-2)^2x_i^2\\[3pt]
&=\frac{(w-2)^2}{4}\sum_{i=1}^4 x_i^2\\[3pt]
&=\frac{(w-2)^2}{4}(1+0+1+4)\\[3pt]
&=\frac32(w-2)^2
=\frac32w^2-6w+6.
\end{aligned}
$$

Derivamos e igualamos a cero:

$$
L'(w)=3w-6=0
\quad\Longrightarrow\quad w^*=2.
$$

Además, $L''(w)=3>0$ en todo $\mathbb{R}$, de modo que el mínimo es global y único. Por sustitución,

$$
\boxed{w^*=2,\qquad L(w^*)=0.}
$$

In [ ]:
x_data = np.array([-1.0, 0.0, 1.0, 2.0])
y_data = np.array([-2.0, 0.0, 2.0, 4.0])

def perdida(w, x=x_data, y=y_data):
    return np.mean((w*x-y)**2)

def gradiente(w, x=x_data, y=y_data):
    return 2*np.mean(x*(w*x-y))

display(Math(r'\boxed{L(2)=%.0f,\qquad L\prime(2)=%.0f}' % (
    perdida(2.0), gradiente(2.0)
)))

## VI.2. Dos actualizaciones batch

Partimos de $w_0=0$ y utilizamos la tasa de aprendizaje $\alpha=0.1$. La regla de descenso es

$$
w_{k+1}=w_k-\alpha L'(w_k).
$$

**Primera actualización.** Al evaluar la derivada en el estado inicial,

$$
\begin{aligned}
L'(w_0)&=3(0)-6=-6,\\
w_1&=0-0.1(-6)=\boxed{0.6}.
\end{aligned}
$$

Las pérdidas antes y después de este paso son

$$
\begin{aligned}
L(w_0)&=\frac32(0-2)^2=6,\\
L(w_1)&=\frac32(0.6-2)^2=2.94.
\end{aligned}
$$

**Segunda actualización.** Recalculamos el gradiente con el nuevo parámetro:

$$
\begin{aligned}
L'(w_1)&=3(0.6)-6=-4.2,\\
w_2&=0.6-0.1(-4.2)=\boxed{1.02},\\
L(w_2)&=\frac32(1.02-2)^2=\boxed{1.4406}.
\end{aligned}
$$

En consecuencia,

$$
\boxed{\begin{aligned}
w_0=0&\quad\longrightarrow\quad w_1=0.6\quad\longrightarrow\quad w_2=1.02,\\[3pt]
L(w_0)=6&\quad\longrightarrow\quad L(w_1)=2.94\quad\longrightarrow\quad L(w_2)=1.4406.
\end{aligned}}
$$

In [ ]:
alpha = 0.1
w = 0.0
iteraciones = [(0, w, perdida(w), gradiente(w))]

for k in range(1, 3):
    w -= alpha*gradiente(w)
    iteraciones.append((k, w, perdida(w), gradiente(w)))

resultado_pasos = pd.DataFrame(iteraciones, columns=["Iteración", "w", "L(w)", "L'(w)"])
display(resultado_pasos.style.format({"w": "{:.6f}", "L(w)": "{:.6f}", "L'(w)": "{:.6f}"}))

## VI.3. Comparación durante 30 épocas

Se implementan ambos métodos con $\alpha=0.1$:

**Descenso batch:** en cada época se calcula el gradiente usando las cuatro observaciones y se realiza **una** actualización.

**Minibatch SGD:** se barajan los datos al comienzo de cada época y se forman dos minibatches de tamaño $2$. Se realizan **dos** actualizaciones por época. Repetimos el experimento con las semillas $0,1,2,3,4$.

La pérdida completa $L(w)$ se registra al final de cada época para comparar las trayectorias sobre una misma métrica.

In [ ]:
def gd_batch(epocas=30, alpha=0.1):
    w = 0.0
    historia = [perdida(w)]

    for _ in range(epocas):
        w = w - alpha*gradiente(w)
        historia.append(perdida(w))

    return w, np.array(historia)


def minibatch_sgd(seed, epocas=30, alpha=0.1, batch_size=2):
    rng = np.random.default_rng(seed)

    w = 0.0
    historia = [perdida(w)]

    for _ in range(epocas):
        indices = rng.permutation(len(x_data))

        for inicio in range(0, len(x_data), batch_size):
            lote = indices[inicio:inicio+batch_size]

            grad = gradiente(
                w,
                x_data[lote],
                y_data[lote]
            )

            w = w - alpha*grad

        historia.append(perdida(w))

    return w, np.array(historia)

In [ ]:
resultados = []
historias = {}

w_batch, hist_batch = gd_batch()
resultados.append(("GD batch", "—", w_batch, perdida(w_batch)))
historias["GD batch"] = hist_batch

for seed in range(5):
    w_sgd, hist_sgd = minibatch_sgd(seed)
    resultados.append(("Minibatch SGD", seed, w_sgd, perdida(w_sgd)))
    historias[f"SGD semilla {seed}"] = hist_sgd

tabla_vi = pd.DataFrame(resultados, columns=["Método", "Semilla", "w final", "L(w) final"])
display(tabla_vi.style.format({"w final": "{:.12f}", "L(w) final": "{:.6e}"}))

In [ ]:
plt.figure(figsize=(9, 6))

for etiqueta, valores in historias.items():
    epocas = np.arange(len(valores))
    plt.semilogy(epocas, valores, label=etiqueta)

plt.xlabel("Época")
plt.ylabel(r"Pérdida completa $L(w)$")
plt.title("VI.3 — GD batch y minibatch SGD")
plt.grid(True)
plt.legend()
plt.show()

Los dos métodos se acercan al mismo valor analítico $w^*=2$. Las diferencias entre las cinco trayectorias de minibatch provienen de los distintos órdenes en que se procesan los datos.

## VI.4. Discusión de los resultados

**(i) ¿Por qué el gradiente de minibatch es insesgado?** Si se elige uniformemente un conjunto $B$ de observaciones y su gradiente es el promedio de los gradientes individuales, cada muestra tiene la misma probabilidad de contribuir. Por linealidad de la esperanza,

$$
\boxed{\mathbb E\bigl[\nabla L_B(w)\bigr]=\nabla L(w).}
$$

**(ii) ¿Por qué la pérdida completa puede no ser monótona?** Cada lote aproxima el gradiente completo con información parcial. En un problema general, una actualización puede disminuir la pérdida del lote sin disminuir la pérdida total. Reducir $\alpha$ cuando el entrenamiento avanza permite hacer ajustes más pequeños y controlar oscilaciones. *En estos datos concretos*, todos los ejemplos no nulos comparten el mismo óptimo $w=2$; por eso las trayectorias observadas son especialmente regulares.

**(iii) ¿La comparación por época demuestra superioridad universal?** No. Batch realiza una actualización usando cuatro observaciones; minibatch realiza dos actualizaciones usando dos observaciones cada una. Aunque una época recorre todos los datos en ambos casos, el costo por actualización y el número de pasos no son iguales. Además, aquí solo se utiliza un conjunto pequeño y particularmente simple.

**(iv) ¿Qué aporta Adam?** Adam combina dos promedios móviles: el **primer momento** del gradiente, que suaviza la dirección de actualización, y el **segundo momento no centrado**, calculado a partir del gradiente al cuadrado, que permite adaptar el tamaño efectivo de paso para cada parámetro.

$$
\begin{aligned}
m_t&=\beta_1m_{t-1}+(1-\beta_1)g_t,\\
v_t&=\beta_2v_{t-1}+(1-\beta_2)g_t^2.
\end{aligned}
$$